# More APIs and Web Scraping Part 1
**Name**: Isabelle Sakelaris \
**Date**: October 12, 2026

In [8]:
import requests
from getpass import getpass # way of protecting the api key so it isn't public in the notebook
from time import sleep # will help us set up a ime delay to match the API's rate limit

key = getpass("Paste your NYT API key: ") 

Paste your NYT API key:  ········


In [14]:
query = "midterms" # search term in any of the metadata

begin_date = "20260901" # yyyymmdd begin date
end_date = "20260930" # yyyymmdd end date

url = f"https://api.nytimes.com/svc/search/v2/articlesearch.json?q={query}&begin_date={begin_date}&end_date={end_date}&api-key={key}"
response = requests.get(url)
print(response.ok)

True


Pulling the actual documents out of the API response:
- first we parse the response as json
- then we find the key that contains teh article info

In [29]:
parsed_nytimes_response = response.json() # interpret the response as python-readable json
print(parsed_nytimes_response['response'].keys())
nyt_docs = parsed_nytimes_response['response']['docs'] # new var with just the info I want
print(nyt_docs[0].keys())
print(nyt_docs[0]['headline']['main'])

dict_keys(['docs', 'metadata'])
dict_keys(['abstract', 'byline', 'document_type', 'headline', '_id', 'keywords', 'multimedia', 'news_desk', 'print_page', 'print_section', 'pub_date', 'section_name', 'snippet', 'source', 'subsection_name', 'type_of_material', 'uri', 'web_url', 'word_count'])
As Midterms Approach, Republicans Find There Is No Escape From Trump


## Pagination

In [ ]:
results = []

for page in range(0, 5): # sets a range of pages starting at 0 and going five times
    url = f"https://api.nytimes.com/svc/search/v2/articlesearch.json?q={query}&begin_date={begin_date}&end_date={end_date}&page={page}&api-key={key}" #adding page to API request and grab the next page and keep doing that until you reach the end of the range
    response = requests.get(url)

    if response.ok:
        parsed = response.json()
        articles = parsed["response"]["docs"]
        results.append(articles)
    else:
        print("page", page, "failed with status code", response.status_code)

    sleep(12)

### Web Scraping

Below are the libraries we need in order to do web scraping

In [30]:
import csv
import json
import os # operating system library, lets us do things like making new folders
import requests
from time import sleep
from bs4 import BeautifulSoup # the scraping library!

html that we will traverse using beautiful soup functions:

In [33]:
html = """
<html>
  <body>
    <h1>Adeline Ravoux</h1>
    <p class="artist">Vincent van Gogh</p>
    <p class="date">1890</p>
    <a href="/art/1958.31">Artwork page</a>
  </body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")  # html.parser is the built-in parser that comes with python

['ASCII_SPACES', 'DEFAULT_BUILDER_FEATURES', 'EMPTY_ELEMENT_EVENT', 'END_ELEMENT_EVENT', 'MAIN_CONTENT_STRING_TYPES', 'ROOT_TAG_NAME', 'START_ELEMENT_EVENT', 'STRING_ELEMENT_EVENT', '_TreeTraversalEvent', '__annotations__', '__bool__', '__call__', '__class__', '__contains__', '__copy__', '__deepcopy__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setitem__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__unicode__', '__weakref__', '_all_strings', '_decode_markup', '_event_stream', '_feed', '_find_all', '_find_one', '_format_tag', '_generator_for_recursive', '_indent_string', '_insert', '_is_root', '_is_xml', '_lastRecursiveChild', '_last_descendant', '_linkage_

You need to figure out how things are unique and then find them

In [34]:
title = soup.find("h1") # find the first element that uses the <h1> tag
print(title)

<h1>Adeline Ravoux</h1>


In [35]:
print(title.get_text()) # only extract the text in the element

Adeline Ravoux


In [43]:
p_elements = soup.find_all("p") # this will give us a list of every matching item
print(p_elements)

[<p class="artist">Vincent van Gogh</p>, <p class="date">1890</p>]


In [50]:
artist = soup.find("p", class_="artist") # the underscore is just a quirk of python, bc "class" is a reserved word
print(artist.get_text())
date = artist.find_next_sibling("p")
print(date.get_text())

Vincent van Gogh
1890


It is useful to look at the front end of the web page to get a visual for how information is organized. You can use inspect mode for this. For this website, the div is called "artwork-details-container".

### Headers

You can identify yourself to the web administrator by setting up a header.

#### Re-use the snippet below to fetch a page!

In [ ]:
headers = {"User-Agent": "INFO664-class-project (educational)"}   # you can call this whatever you want, it shows up on the scraped website's logs as who you are (ie not a Google/AI bot or web browser). 

os.makedirs("scraped_pages", exist_ok=True)  # make a new folder to hold our scraped website HTML data. exist_ok = True means, if the directory already exists, do nothing and don't return an error

def fetch_page(url, filename):  # declaring a new function, which takes a URL to be scraped, and a destination filename, as arguments
    if os.path.exists(filename):  # check to see if file already exists, if so, don't download it from the web, load it instead
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()  # in this case, return provides the read file data, and escapes from the function, ignoring everything below

    response = requests.get(url, headers=headers) # grab the file, using our pre-defined headers as the header data

    if not response.ok: # if the response reports an error
        print(url, "failed with status code", response.status_code) # tell us why it failed
        return None # return a "None" object, since there's nothing usable in the response, and escape from this function
    with open(filename, "w",  encoding="utf-8") as file:
        file.write(response.text)

    sleep(10) # wait 10 seconds, since our robots.txt file asks for a 10 second Crawl-delay
    return response.text

### Robots.txt permissions
We can view the permissions a website grants to do things like accessing data with a bot by looking at its robots.txt file. The robots.txt file always lives in the root directory of the domain, like `clevelandart.org/robots/txt`.

For example, `user-agent: *` = any user agent can access this site. There is also a ten-second crawl delay.

At this point, in order to not hit the rate limit, we just observed the scraping itself. There is a link to the zoom recording on our canvas site.

In [53]:
artwork_details = soup.find(id="artwork-details-container")
# print(artwork_details.find_all(True)) # will print the whole page
children = artwork_details.find_all(True, recursive=False) # recursvie=false does one layer of children

for index, child in enumerate(children):
    print(index, child.name, "|", child.get_text()) # prints the index number, the name of the element, a separating pipe, and the text

AttributeError: 'NoneType' object has no attribute 'find_all'